In [ ]:
import torch.nn as nn
import torch

In [ ]:
class SeparableConv2d(nn.Module):
    def __init__(self, input_channels, output_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        self.depthwise_conv = nn.Conv2d(
            in_channels=input_channels, out_channels=input_channels, kernel_size=kernel_size,
            stride=stride, padding=padding, groups=input_channels
        )
        self.pointwise_conv = nn.Conv2d(
            in_channels=input_channels, out_channels=output_channels, kernel_size=1, stride=1, padding=0
        )

    def forward(self, inputs):
        return self.pointwise_conv(self.depthwise_conv(inputs))

In [ ]:
class Xception(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()
        self.entry = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )

        self.middle = nn.Sequential(
            SeparableConv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            SeparableConv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            SeparableConv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
        )

        self.exit = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.entry(x)
        x = self.middle(x)
        x = self.exit(x)
        return x